# DFS vs. BFS — A Comparative Analysis
**COMP 3305 Class Project — Bryce Anderson**

Run all cells (Runtime → Run all).

## 1. Implementation

In [ ]:
"""
COMP 3305 Class Project - Depth-First Search vs. Breadth-First Search
Author: Bryce Anderson

Implements BFS and DFS (iterative and recursive) on an adjacency-list graph,
plus instrumentation used for the comparative analysis:
  - order in which vertices are visited
  - number of vertices explored before reaching a target
  - peak frontier size (queue/stack) -> memory behaviour
  - length of the path found (BFS = shortest in unweighted graphs)
"""
from collections import deque
import sys


class Graph:
    """Unweighted graph stored as an adjacency list (dict: vertex -> list)."""

    def __init__(self, directed=False):
        self.adj = {}
        self.directed = directed

    def add_vertex(self, v):
        self.adj.setdefault(v, [])

    def add_edge(self, u, v):
        self.add_vertex(u)
        self.add_vertex(v)
        self.adj[u].append(v)
        if not self.directed:
            self.adj[v].append(u)

    def neighbors(self, v):
        return self.adj.get(v, [])

    def __len__(self):
        return len(self.adj)


# ---------------------------------------------------------------------------
# Breadth-First Search  -  FIFO queue, explores level by level
# ---------------------------------------------------------------------------
def bfs(graph, start, target=None):
    """Return dict with visit order, parent map, explored count, peak frontier."""
    visited = {start}
    parent = {start: None}
    queue = deque([start])
    order = []
    peak = 1
    while queue:
        u = queue.popleft()
        order.append(u)
        if u == target:
            break
        for w in graph.neighbors(u):
            if w not in visited:
                visited.add(w)
                parent[w] = u
                queue.append(w)
        peak = max(peak, len(queue))
    return {"order": order, "parent": parent, "explored": len(order),
            "peak_frontier": peak, "found": target in parent if target is not None else None}


# ---------------------------------------------------------------------------
# Depth-First Search  -  LIFO stack, goes as deep as possible first
# ---------------------------------------------------------------------------
def dfs(graph, start, target=None):
    """Iterative DFS (explicit stack). Visits neighbours in the same order as
    the recursive version so both produce identical traversal orders."""
    visited = set()
    parent = {start: None}
    stack = [start]
    order = []
    peak = 1
    while stack:
        u = stack.pop()
        if u in visited:
            continue
        visited.add(u)
        order.append(u)
        if u == target:
            break
        for w in reversed(graph.neighbors(u)):
            if w not in visited:
                parent[w] = u
                stack.append(w)
        peak = max(peak, len(stack))
    return {"order": order, "parent": parent, "explored": len(order),
            "peak_frontier": peak, "found": target in visited if target is not None else None}


def dfs_recursive(graph, start, visited=None, order=None):
    """Classic recursive DFS (limited by Python's recursion depth)."""
    if visited is None:
        visited, order = set(), []
    visited.add(start)
    order.append(start)
    for w in graph.neighbors(start):
        if w not in visited:
            dfs_recursive(graph, w, visited, order)
    return order


def reconstruct_path(parent, target):
    if target not in parent:
        return None
    path = []
    while target is not None:
        path.append(target)
        target = parent[target]
    return path[::-1]


# ---------------------------------------------------------------------------
# Applications built on top of BFS / DFS
# ---------------------------------------------------------------------------
def has_cycle_directed(graph):
    """DFS with 3 colours: a back edge to a GRAY vertex means a cycle."""
    WHITE, GRAY, BLACK = 0, 1, 2
    color = {v: WHITE for v in graph.adj}

    def visit(u):
        color[u] = GRAY
        for w in graph.neighbors(u):
            if color[w] == GRAY or (color[w] == WHITE and visit(w)):
                return True
        color[u] = BLACK
        return False

    return any(color[v] == WHITE and visit(v) for v in graph.adj)


def topological_sort(graph):
    """DFS-based topological order of a DAG."""
    seen, out = set(), []

    def visit(u):
        seen.add(u)
        for w in graph.neighbors(u):
            if w not in seen:
                visit(w)
        out.append(u)

    for v in graph.adj:
        if v not in seen:
            visit(v)
    return out[::-1]


def connected_components(graph):
    """Counts components by running BFS from every unvisited vertex."""
    seen, comps = set(), []
    for v in graph.adj:
        if v not in seen:
            comp = bfs(graph, v)["order"]
            seen.update(comp)
            comps.append(comp)
    return comps


# ---------------------------------------------------------------------------
# Graph generators used in the experiments
# ---------------------------------------------------------------------------
def grid_graph(n, walls=frozenset()):
    """n x n grid maze; vertices are (row, col); walls are blocked cells."""
    g = Graph()
    for r in range(n):
        for c in range(n):
            if (r, c) in walls:
                continue
            g.add_vertex((r, c))
            for dr, dc in ((0, 1), (1, 0)):
                rr, cc = r + dr, c + dc
                if rr < n and cc < n and (rr, cc) not in walls:
                    g.add_edge((r, c), (rr, cc))
    return g


def random_graph(n, avg_degree, seed=0):
    import random
    rnd = random.Random(seed)
    g = Graph()
    for v in range(n):
        g.add_vertex(v)
    for v in range(1, n):  # spanning tree keeps the graph connected
        g.add_edge(v, rnd.randrange(v))
    extra = n * avg_degree // 2 - (n - 1)
    for _ in range(max(0, extra)):
        g.add_edge(rnd.randrange(n), rnd.randrange(n))
    return g


def complete_tree(branching, depth):
    """Complete tree: every internal node has `branching` children."""
    g = Graph()
    g.add_vertex(0)
    frontier, nxt = [0], 1
    for _ in range(depth):
        new = []
        for u in frontier:
            for _ in range(branching):
                g.add_edge(u, nxt)
                new.append(nxt)
                nxt += 1
        frontier = new
    return g


def sample_graph():
    """Small 8-vertex graph used on the slides and in the demo."""
    g = Graph()
    for u, v in [("A", "B"), ("A", "C"), ("B", "D"), ("B", "E"),
                 ("C", "F"), ("C", "G"), ("C", "H"), ("E", "H"), ("F", "H")]:
        g.add_edge(u, v)
    return g


if True:  # demo
    sys.setrecursionlimit(10000)
    g = sample_graph()
    print("Sample graph adjacency list:")
    for v, ns in g.adj.items():
        print(f"  {v}: {ns}")
    print("\nBFS order from A:          ", " -> ".join(bfs(g, "A")["order"]))
    print("DFS order from A:          ", " -> ".join(dfs(g, "A")["order"]))
    print("DFS (recursive) from A:    ", " -> ".join(dfs_recursive(g, "A")))
    rb, rd = bfs(g, "A", "H"), dfs(g, "A", "H")
    print("\nPath A->H found by BFS:", reconstruct_path(rb["parent"], "H"))
    print("Path A->H found by DFS:", reconstruct_path(rd["parent"], "H"))

    dag = Graph(directed=True)
    for u, v in [("Intro CS", "Data Structures"), ("Discrete Math", "Algorithms"),
                 ("Data Structures", "Algorithms"), ("Algorithms", "AI")]:
        dag.add_edge(u, v)
    print("\nDFS application - course order (topological sort):", topological_sort(dag))
    print("Cycle in course graph?", has_cycle_directed(dag))


## 2. Visualize the traversal order on the sample graph

In [ ]:
import networkx as nx, matplotlib.pyplot as plt
g = sample_graph()
G = nx.Graph([(u, v) for u in g.adj for v in g.adj[u]])
pos = {"A":(0,3),"B":(-1.5,2),"C":(1.5,2),"D":(-2.2,1),"E":(-0.8,1),"F":(0.8,1),"G":(2.2,1),"H":(0,0)}
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, order) in zip(axes, [("BFS", bfs(g, "A")["order"]), ("DFS", dfs(g, "A")["order"])]):
    nx.draw(G, pos, ax=ax, with_labels=True, node_color="#f2c94c", node_size=900, font_weight="bold")
    for i, v in enumerate(order, 1):
        ax.annotate(str(i), pos[v], xytext=(18, 12), textcoords="offset points", color="#5b2c83", fontsize=13, fontweight="bold")
    ax.set_title(f"{name} visit order: " + " → ".join(order))
plt.show()

## 3. Comparative experiments

In [ ]:
"""
Comparative analysis of BFS vs DFS.

Experiment 1 - Runtime scaling on random connected graphs (avg degree 6):
               both should grow linearly in V + E.
Experiment 2 - Open n x n grid, search from the top-left corner to the
               centre and to the far corner: vertices explored, path length
               found, and peak frontier size (memory).
Experiment 3 - Complete binary tree: peak frontier size during a full
               traversal (BFS holds a whole level, DFS only one branch).
Results are printed as tables and saved to results.json.
"""
import json
import time
from statistics import median




def time_it(fn, *args, repeats=5):
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn(*args)
        times.append(time.perf_counter() - t0)
    return median(times) * 1000  # ms


def experiment_runtime(sizes=(10_000, 50_000, 100_000, 200_000, 400_000)):
    rows = []
    for n in sizes:
        g = random_graph(n, avg_degree=6, seed=42)
        edges = sum(len(v) for v in g.adj.values()) // 2
        rows.append({"V": n, "E": edges,
                     "bfs_ms": round(time_it(bfs, g, 0), 1),
                     "dfs_ms": round(time_it(dfs, g, 0), 1)})
    return rows


def experiment_grid(n=101):
    g = grid_graph(n)
    start = (0, 0)
    out = {}
    for label, target in (("near", (n // 10, n // 10)),
                          ("centre", (n // 2, n // 2)),
                          ("far corner", (n - 1, n - 1))):
        row = {}
        for name, algo in (("BFS", bfs), ("DFS", dfs)):
            r = algo(g, start, target)
            path = reconstruct_path(r["parent"], target)
            row[name] = {"explored": r["explored"],
                         "path_len": len(path) - 1,
                         "peak_frontier": r["peak_frontier"]}
        row["optimal"] = abs(target[0]) + abs(target[1])
        out[label] = row
    return out


def experiment_tree(branching=2, depths=(8, 10, 12, 14, 16, 18)):
    """Full traversal of a complete binary tree: peak frontier = memory."""
    rows = []
    for d in depths:
        g = complete_tree(branching, d)
        rows.append({"depth": d, "V": len(g),
                     "bfs_peak": bfs(g, 0)["peak_frontier"],
                     "dfs_peak": dfs(g, 0)["peak_frontier"]})
    return rows


if True:
    print("Experiment 1: runtime on random graphs (median of 5 runs)")
    print(f"{'V':>8} {'E':>9} {'BFS ms':>9} {'DFS ms':>9}")
    rt = experiment_runtime()
    for r in rt:
        print(f"{r['V']:>8} {r['E']:>9} {r['bfs_ms']:>9} {r['dfs_ms']:>9}")

    n = 101
    print(f"\nExperiment 2: {n}x{n} open grid ({n*n} vertices), start (0,0)")
    grid = experiment_grid(n)
    for label, row in grid.items():
        print(f"  target {label} (optimal path = {row['optimal']} steps)")
        for name in ("BFS", "DFS"):
            d = row[name]
            print(f"    {name}: explored {d['explored']:>6}  path {d['path_len']:>6}  "
                  f"peak frontier {d['peak_frontier']:>6}")

    print("\nExperiment 3: complete binary tree, full traversal - peak frontier (memory)")
    tree = experiment_tree()
    print(f"{'depth':>6} {'V':>8} {'BFS peak':>9} {'DFS peak':>9}")
    for r in tree:
        print(f"{r['depth']:>6} {r['V']:>8} {r['bfs_peak']:>9} {r['dfs_peak']:>9}")

    with open("results.json", "w") as f:
        json.dump({"runtime": rt, "grid": grid, "grid_n": n, "tree": tree}, f, indent=2)
    print("\nSaved results.json")


## 4. Plot the results

In [ ]:
rt = experiment_runtime()
tree = experiment_tree()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
a1.plot([r["V"] for r in rt], [r["bfs_ms"] for r in rt], "o-", label="BFS")
a1.plot([r["V"] for r in rt], [r["dfs_ms"] for r in rt], "s-", label="DFS")
a1.set(title="Runtime grows linearly: O(V + E)", xlabel="vertices (avg degree 6)", ylabel="ms"); a1.legend()
a2.semilogy([r["depth"] for r in tree], [r["bfs_peak"] for r in tree], "o-", label="BFS queue")
a2.semilogy([r["depth"] for r in tree], [r["dfs_peak"] for r in tree], "s-", label="DFS stack")
a2.set(title="Peak memory on a complete binary tree", xlabel="tree depth", ylabel="frontier size (log)"); a2.legend()
plt.show()